# LeetCode 3194 — Minimum Average of Smallest and Largest Elements bằng Python

Link đề: https://leetcode.com/problems/minimum-average-of-smallest-and-largest-elements/ · Độ khó: **Easy** · Chủ đề: Array, Two Pointers, Sorting

Skeleton LeetCode:

```python
from typing import List


class Solution:
    def minimumAverage(self, nums: List[int]) -> float:
        ...
```

Ghi chú: đề và ví dụ được chép/dịch theo đề gốc; mọi đáp án trong notebook đã được kiểm chứng bằng code chạy thật.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: lặp `n / 2` lần: **lấy ra** số nhỏ nhất và số lớn nhất của mảng, ghi lại **trung bình** của hai số đó; cuối cùng trả về **trung bình nhỏ nhất** trong các trung bình đã ghi.
2. Nắm các công cụ Python: `min`, `max`, `list.remove`, `sorted` / `list.sort`, **hai con trỏ** `left`/`right`, **chỉ số âm / đối xứng** `nums[n - 1 - i]`, `zip` với `reversed`, `collections.deque`, và phép chia thực `/` (khác `//`).
3. Đi từ cách **mô phỏng đúng như đề** → **sắp xếp rồi hai con trỏ** → **chỉ so sánh tổng** (chia `2` một lần) → **`deque`** → **đếm theo giá trị** (vì `nums[i] <= 50`).
4. Hiểu mấu chốt: sau khi **sắp xếp**, số nhỏ nhất và lớn nhất ở **hai đầu mảng**; bước `k` lấy phần tử thứ `k` từ đầu và thứ `k` từ cuối, tức là ghép **`nums[i]` với `nums[n - 1 - i]`**.
5. Tránh các lỗi kinh điển: dùng `//` (mất phần `.5`), gọi `sorted(nums)` mà **không gán lại**, ghép sai cặp (ví dụ `nums[i]` với `nums[i + n/2]`), lấy `max` thay vì `min` của các trung bình, và quên rằng đề cần số thực (`float`).

## 2. Đề bài & ví dụ

Có một mảng số thực `averages` ban đầu **rỗng**. Cho mảng `nums` gồm `n` số nguyên, trong đó `n` **chẵn**.

Lặp lại quy trình sau `n / 2` lần:

- **Xoá** phần tử **nhỏ nhất** `minElement` và phần tử **lớn nhất** `maxElement` khỏi `nums`.
- Thêm `(minElement + maxElement) / 2` vào `averages`.

Trả về phần tử **nhỏ nhất** của `averages`.

**Ví dụ 1**

- Input: `nums = [7,8,3,4,15,13,4,1]`
- Output: `5.5`
- Giải thích: bước 1 xoá `1` và `15` → `8`; bước 2 xoá `3` và `13` → `8`; bước 3 xoá `4` và `8` → `6`; bước 4 xoá `4` và `7` → `5.5`. `averages = [8, 8, 6, 5.5]`, nhỏ nhất là `5.5`.

**Ví dụ 2**

- Input: `nums = [1,9,8,3,10,5]`
- Output: `5.5`
- Giải thích: `averages = [5.5, 6, 6.5]`, nhỏ nhất `5.5`.

**Ví dụ 3**

- Input: `nums = [1,2,3,7,8,9]`
- Output: `5.0`
- Giải thích: `averages = [5, 5, 5]`, nhỏ nhất `5.0`.

**Ràng buộc:** `2 <= n == nums.length <= 50`; `n` chẵn; `1 <= nums[i] <= 50`.

In [1]:
import random
from collections import deque
from typing import List

examples = [
    ([7, 8, 3, 4, 15, 13, 4, 1], 5.5),
    ([1, 9, 8, 3, 10, 5], 5.5),
    ([1, 2, 3, 7, 8, 9], 5.0),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[7, 8, 3, 4, 15, 13, 4, 1] -> kỳ vọng 5.5
[1, 9, 8, 3, 10, 5] -> kỳ vọng 5.5
[1, 2, 3, 7, 8, 9] -> kỳ vọng 5.0


## 3. Kiến thức nền

### 3.1. `min`, `max` và `list.remove`

`min(lst)` / `max(lst)` trả phần tử nhỏ / lớn nhất (quét cả list: `O(n)`). `lst.remove(x)` xoá **lần xuất hiện đầu tiên** của giá trị `x` (cũng `O(n)`). Nếu `x` không có, báo `ValueError`.

In [2]:
nums = [7, 8, 3, 4, 15, 13, 4, 1]
print(min(nums), max(nums))              # 1 15
nums.remove(1)
nums.remove(15)
print(nums)                              # [7, 8, 3, 4, 13, 4]   (đã bỏ 1 và 15)
nums2 = [4, 4, 9]
nums2.remove(4)                          # chỉ bỏ MỘT số 4 (số đầu tiên)
print(nums2)                             # [4, 9]

1 15
[7, 8, 3, 4, 13, 4]
[4, 9]


### 3.2. Sắp xếp: `sorted` và `list.sort`

- `sorted(lst)` trả về **list mới** đã sắp xếp, **không** đổi `lst`.
- `lst.sort()` sắp xếp **tại chỗ** và trả `None`.

Sau khi sắp xếp tăng dần, `nums[0]` là nhỏ nhất, `nums[-1]` là lớn nhất.

In [3]:
nums = [3, 1, 2]
print(sorted(nums), nums)                # [1, 2, 3] [3, 1, 2]   (nums chưa đổi)
nums.sort()
print(nums)                              # [1, 2, 3]
print(nums.sort() is None)               # True  (sort() trả về None, đừng viết x = nums.sort())

[1, 2, 3] [3, 1, 2]
[1, 2, 3]
True


### 3.3. Chỉ số từ cuối: `nums[-1]` và `nums[n - 1 - i]`

`nums[-1]` là phần tử cuối; `nums[-2]` là kế cuối. Phần tử **đối xứng** của vị trí `i` qua tâm là vị trí `n - 1 - i`. Ví dụ `n = 6`: `0 ↔ 5`, `1 ↔ 4`, `2 ↔ 3`.

In [4]:
nums = [10, 20, 30, 40, 50, 60]
n = len(nums)
for i in range(n // 2):
    print(i, "<->", n - 1 - i, ":", nums[i], nums[n - 1 - i])
# 0 <-> 5 : 10 60 / 1 <-> 4 : 20 50 / 2 <-> 3 : 30 40
print(nums[-1], nums[-2])                # 60 50

0 <-> 5 : 10 60
1 <-> 4 : 20 50
2 <-> 3 : 30 40
60 50


### 3.4. Hai con trỏ `left`, `right`

Đặt `left = 0` (đầu) và `right = n - 1` (cuối), xử lý cặp `(nums[left], nums[right])` rồi `left += 1`, `right -= 1` cho đến khi gặp nhau. Với `n` chẵn thì kết thúc khi `left > right`.

In [5]:
nums = [1, 2, 3, 7, 8, 9]
left, right = 0, len(nums) - 1
while left < right:
    print(f"left={left}, right={right}: cặp ({nums[left]}, {nums[right]})")
    left += 1
    right -= 1

left=0, right=5: cặp (1, 9)
left=1, right=4: cặp (2, 8)
left=2, right=3: cặp (3, 7)


### 3.5. Chia thực `/` và số thực

`/` luôn cho `float` (`11 / 2 = 5.5`, `10 / 2 = 5.0`); còn `//` chia lấy phần nguyên (`11 // 2 = 5`, mất `.5`). Đề yêu cầu trung bình thực nên dùng `/`. Các số dạng `k / 2` **biểu diễn chính xác** trong `float`, nên so sánh `==` an toàn.

In [6]:
print(11 / 2, 10 / 2, 11 // 2)           # 5.5 5.0 5
print(type(10 / 2).__name__)             # float
print((1 + 15) / 2 == 8.0, 11 / 2 == 5.5)    # True True

5.5 5.0 5
float
True True


### 3.6. `zip` và `reversed`; `collections.deque`

`zip(a, b)` ghép từng cặp phần tử cùng vị trí và dừng khi một list hết. `reversed(lst)` duyệt ngược. `deque` là hàng đợi hai đầu: `popleft()` lấy đầu, `pop()` lấy cuối, đều `O(1)`.

In [7]:
nums = [1, 2, 3, 7, 8, 9]
print(list(zip(nums, reversed(nums))))            # [(1, 9), (2, 8), (3, 7), (7, 3), (8, 2), (9, 1)]
dq = deque(nums)
print(dq.popleft(), dq.pop(), list(dq))           # 1 9 [2, 3, 7, 8]

[(1, 9), (2, 8), (3, 7), (7, 3), (8, 2), (9, 1)]
1 9 [2, 3, 7, 8]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Lấy `nums = [1, 9, 8, 3, 10, 5]`. Hãy **sắp xếp** trước: `[1, 3, 5, 8, 9, 10]`. Giờ làm đúng như đề:

| bước | nhỏ nhất | lớn nhất | trung bình | mảng còn lại |
|---|---|---|---|---|
| 1 | 1 | 10 | 5.5 | `[3, 5, 8, 9]` |
| 2 | 3 | 9 | 6.0 | `[5, 8]` |
| 3 | 5 | 8 | 6.5 | `[]` |

Trung bình nhỏ nhất: `5.5`.

Nhìn bảng: ở bước `k` ta lấy **phần tử thứ `k` từ đầu** và **phần tử thứ `k` từ cuối** của mảng đã sắp xếp. Vậy chỉ cần **sắp xếp một lần** rồi ghép `nums[i]` với `nums[n - 1 - i]`, không cần tìm min/max lại nhiều lần.

Thêm một nhận xét: trung bình `(a + b) / 2` có cùng thứ tự với tổng `a + b`, nên tìm **tổng nhỏ nhất** rồi chia `2` một lần ở cuối.

In [8]:
nums = sorted([1, 9, 8, 3, 10, 5])
n = len(nums)
for i in range(n // 2):
    a, b = nums[i], nums[n - 1 - i]
    print(f"bước {i + 1}: ({a} + {b}) / 2 = {(a + b) / 2}")

bước 1: (1 + 10) / 2 = 5.5
bước 2: (3 + 9) / 2 = 6.0
bước 3: (5 + 8) / 2 = 6.5


## 5. Cách 1 — Brute Force / baseline: mô phỏng đúng như đề

**Ý tưởng.** Làm đúng từng chữ: lặp `n / 2` lần; mỗi lần dùng `min(nums)` và `max(nums)` để tìm hai số, `remove` cả hai khỏi `nums`, thêm trung bình vào `averages`. Cuối cùng trả `min(averages)`.

In [9]:
def minimum_average_simulate(nums: List[int]) -> float:
    nums = list(nums)                    # sao chép để không làm hỏng input gốc
    averages = []
    for _ in range(len(nums) // 2):
        mn = min(nums)                   # quét tìm nhỏ nhất
        mx = max(nums)                   # quét tìm lớn nhất
        nums.remove(mn)                  # bỏ một số nhỏ nhất
        nums.remove(mx)                  # bỏ một số lớn nhất
        averages.append((mn + mx) / 2)
    return min(averages)


for nums, expected in examples:
    print(minimum_average_simulate(nums), "(kỳ vọng", expected, ")")

5.5 (kỳ vọng 5.5 )
5.5 (kỳ vọng 5.5 )
5.0 (kỳ vọng 5.0 )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Có `n/2` vòng; mỗi vòng `min`, `max`, hai lần `remove` đều `O(n)` ⇒ **Time `O(n²)`**, **Space `O(n)`** (bản sao và `averages`). Với `n <= 50` rất nhanh.

**Nút thắt:** mỗi vòng ta **quét lại cả mảng** để tìm min và max. Sau khi sắp xếp, hai đầu mảng chính là min và max, không cần quét.

## 6. Ý tưởng thuật toán / bước đột phá

**Sắp xếp một lần.** Sau `nums.sort()`:

- vòng `1` lấy `nums[0]` và `nums[n-1]`;
- vòng `2` lấy `nums[1]` và `nums[n-2]` (vì hai số trước đã bị xoá);
- ...
- vòng `k` lấy `nums[k-1]` và `nums[n-k]`.

Mỗi vòng hai con trỏ `left`, `right` tiến vào trong một ô. Có tổng `n/2` cặp. Tổng thời gian là chi phí sắp xếp `O(n log n)` cộng quét `O(n)`.

Thêm bước tối ưu nhỏ: chỉ cần theo dõi **tổng nhỏ nhất** `best = min(nums[i] + nums[n-1-i])`, rồi trả `best / 2` (chỉ chia một lần).

In [10]:
nums = sorted([7, 8, 3, 4, 15, 13, 4, 1])
print(nums)                              # [1, 3, 4, 4, 7, 8, 13, 15]
left, right = 0, len(nums) - 1
best = float("inf")
while left < right:
    s = nums[left] + nums[right]
    best = min(best, s)
    print(f"cặp ({nums[left]}, {nums[right]}): tổng {s}, tổng nhỏ nhất đến giờ {best}")
    left += 1
    right -= 1
print("đáp án =", best / 2)              # 5.5

[1, 3, 4, 4, 7, 8, 13, 15]
cặp (1, 15): tổng 16, tổng nhỏ nhất đến giờ 16
cặp (3, 13): tổng 16, tổng nhỏ nhất đến giờ 16
cặp (4, 8): tổng 12, tổng nhỏ nhất đến giờ 12
cặp (4, 7): tổng 11, tổng nhỏ nhất đến giờ 11
đáp án = 5.5


## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp + hai con trỏ, lưu `averages`

**Giải thích từng đoạn.**

1. `nums = sorted(nums)` (hoặc `nums.sort()`): đầu mảng là nhỏ nhất, cuối là lớn nhất.
2. `left, right = 0, n - 1`.
3. Khi `left < right`: thêm `(nums[left] + nums[right]) / 2` vào `averages`, rồi `left += 1`, `right -= 1`.
4. Trả `min(averages)`.

In [11]:
def minimum_average_two_pointers(nums: List[int]) -> float:
    nums = sorted(nums)                  # sorted trả list mới: phải GÁN lại
    averages = []
    left, right = 0, len(nums) - 1
    while left < right:
        averages.append((nums[left] + nums[right]) / 2)
        left += 1
        right -= 1
    return min(averages)


for nums, expected in examples:
    print(minimum_average_two_pointers(nums), "(kỳ vọng", expected, ")")

5.5 (kỳ vọng 5.5 )
5.5 (kỳ vọng 5.5 )
5.0 (kỳ vọng 5.0 )


**Độ phức tạp Cách 2:** sắp xếp `O(n log n)`, quét `O(n)` ⇒ **Time `O(n log n)`**, **Space `O(n)`** (`sorted` tạo list mới và `averages`).

### Cách 3 — Sắp xếp + `min` các tổng, chia `2` một lần

**Giải thích.** Vì `(a + b) / 2` tăng theo `a + b`, trung bình nhỏ nhất ứng với tổng nhỏ nhất. Duyệt `i` từ `0` đến `n // 2 - 1`, lấy `min(nums[i] + nums[n - 1 - i])`, rồi trả `best / 2`. Không cần list `averages`, tránh phép chia lặp.

In [12]:
def minimum_average_min_sum(nums: List[int]) -> float:
    nums = sorted(nums)
    n = len(nums)
    best = min(nums[i] + nums[n - 1 - i] for i in range(n // 2))    # tổng nhỏ nhất của các cặp đối xứng
    return best / 2


for nums, expected in examples:
    print(minimum_average_min_sum(nums), "(kỳ vọng", expected, ")")

5.5 (kỳ vọng 5.5 )
5.5 (kỳ vọng 5.5 )
5.0 (kỳ vọng 5.0 )


In [13]:
# Code nộp LeetCode cho Cách 3 (đổi tên lớp thành Solution khi nộp)
class SolutionMinSum:
    def minimumAverage(self, nums: List[int]) -> float:
        nums.sort()
        n = len(nums)
        return min(nums[i] + nums[n - 1 - i] for i in range(n // 2)) / 2


print(SolutionMinSum().minimumAverage([1, 9, 8, 3, 10, 5]))       # 5.5

5.5


**Độ phức tạp Cách 3:** **Time `O(n log n)`**, **Space `O(1)` phụ** nếu `nums.sort()` tại chỗ (`O(n)` nếu dùng `sorted`).

### Cách 4 — `deque` sau khi sắp xếp: `popleft` và `pop`

**Giải thích.** Sắp xếp rồi đưa vào `deque`; mỗi vòng `popleft()` (nhỏ nhất) và `pop()` (lớn nhất) — **đúng từng chữ của đề** (xoá hai phần tử), nhưng mỗi thao tác chỉ `O(1)`. Cách này phản ánh sát mô tả "remove min and max".

In [14]:
def minimum_average_deque(nums: List[int]) -> float:
    dq = deque(sorted(nums))
    best = float("inf")
    while dq:
        mn = dq.popleft()                # nhỏ nhất hiện tại
        mx = dq.pop()                    # lớn nhất hiện tại
        best = min(best, (mn + mx) / 2)
    return best


for nums, expected in examples:
    print(minimum_average_deque(nums), "(kỳ vọng", expected, ")")

5.5 (kỳ vọng 5.5 )
5.5 (kỳ vọng 5.5 )
5.0 (kỳ vọng 5.0 )


**Độ phức tạp Cách 4:** **Time `O(n log n)`**, **Space `O(n)`**.

### Cách 5 — Đếm theo giá trị (counting sort) vì `nums[i] <= 50`

**Giải thích.** Vì giá trị chỉ từ `1` đến `50`, ta thay `sort` bằng **bảng đếm** `cnt[v]` = số lần xuất hiện của `v`. Duyệt `v = 0..50` để dựng lại danh sách đã sắp xếp (`v` lặp `cnt[v]` lần): `O(n + V)` với `V = 51`. Rồi ghép đối xứng như Cách 3. Ý tưởng này quan trọng khi giá trị **bị chặn nhỏ**: sắp xếp tuyến tính.

In [15]:
def minimum_average_counting(nums: List[int]) -> float:
    cnt = [0] * 51                       # cnt[v]: số lần xuất hiện của giá trị v (1 <= v <= 50)
    for x in nums:
        cnt[x] += 1
    ordered = []
    for v in range(51):
        ordered.extend([v] * cnt[v])     # v lặp cnt[v] lần -> danh sách đã sắp xếp
    n = len(ordered)
    best = min(ordered[i] + ordered[n - 1 - i] for i in range(n // 2))
    return best / 2


for nums, expected in examples:
    print(minimum_average_counting(nums), "(kỳ vọng", expected, ")")

5.5 (kỳ vọng 5.5 )
5.5 (kỳ vọng 5.5 )
5.0 (kỳ vọng 5.0 )


**Độ phức tạp Cách 5:** **Time `O(n + V)`** với `V = 51` (coi là `O(n)`), **Space `O(n + V)`**.

## 8. TRACE từng bước

Ta trace Cách 1 (mô phỏng) trên `nums = [1, 9, 8, 3, 10, 5]` và Cách 2 (hai con trỏ) trên `nums = [7, 8, 3, 4, 15, 13, 4, 1]`.

Bảng trace Cách 2 với `nums = [7, 8, 3, 4, 15, 13, 4, 1]` (sau sắp xếp: `[1, 3, 4, 4, 7, 8, 13, 15]`):

| bước | left | right | `nums[left]` | `nums[right]` | trung bình | `averages` |
|---|---|---|---|---|---|---|
| 1 | 0 | 7 | 1 | 15 | 8.0 | `[8.0]` |
| 2 | 1 | 6 | 3 | 13 | 8.0 | `[8.0, 8.0]` |
| 3 | 2 | 5 | 4 | 8 | 6.0 | `[8.0, 8.0, 6.0]` |
| 4 | 3 | 4 | 4 | 7 | 5.5 | `[8.0, 8.0, 6.0, 5.5]` |

Hết vòng (`left = 4`, `right = 3`, `left < right` sai). Đáp án `min = 5.5`.

In [16]:
def trace_simulate(nums: List[int]) -> float:
    nums = list(nums)
    averages = []
    print(f"nums = {nums}")
    step = 0
    while nums:
        step += 1
        mn, mx = min(nums), max(nums)
        nums.remove(mn)
        nums.remove(mx)
        averages.append((mn + mx) / 2)
        print(f"  bước {step}: min={mn}, max={mx}, nums còn {nums}, averages = {averages}")
    print("  đáp án =", min(averages))
    return min(averages)


assert trace_simulate([1, 9, 8, 3, 10, 5]) == 5.5

nums = [1, 9, 8, 3, 10, 5]
  bước 1: min=1, max=10, nums còn [9, 8, 3, 5], averages = [5.5]
  bước 2: min=3, max=9, nums còn [8, 5], averages = [5.5, 6.0]
  bước 3: min=5, max=8, nums còn [], averages = [5.5, 6.0, 6.5]
  đáp án = 5.5


In [17]:
def trace_two_pointers(nums: List[int]) -> float:
    nums = sorted(nums)
    averages = []
    left, right = 0, len(nums) - 1
    print(f"sau khi sắp xếp: {nums}")
    step = 0
    while left < right:
        step += 1
        avg = (nums[left] + nums[right]) / 2
        averages.append(avg)
        print(f"  bước {step}: left={left}, right={right}, ({nums[left]} + {nums[right]}) / 2 = {avg}, averages = {averages}")
        left += 1
        right -= 1
    print(f"  dừng: left={left}, right={right}; đáp án = {min(averages)}")
    return min(averages)


assert trace_two_pointers([7, 8, 3, 4, 15, 13, 4, 1]) == 5.5
assert trace_two_pointers([1, 2, 3, 7, 8, 9]) == 5.0

sau khi sắp xếp: [1, 3, 4, 4, 7, 8, 13, 15]
  bước 1: left=0, right=7, (1 + 15) / 2 = 8.0, averages = [8.0]
  bước 2: left=1, right=6, (3 + 13) / 2 = 8.0, averages = [8.0, 8.0]
  bước 3: left=2, right=5, (4 + 8) / 2 = 6.0, averages = [8.0, 8.0, 6.0]
  bước 4: left=3, right=4, (4 + 7) / 2 = 5.5, averages = [8.0, 8.0, 6.0, 5.5]
  dừng: left=4, right=3; đáp án = 5.5
sau khi sắp xếp: [1, 2, 3, 7, 8, 9]
  bước 1: left=0, right=5, (1 + 9) / 2 = 5.0, averages = [5.0]
  bước 2: left=1, right=4, (2 + 8) / 2 = 5.0, averages = [5.0, 5.0]
  bước 3: left=2, right=3, (3 + 7) / 2 = 5.0, averages = [5.0, 5.0, 5.0]
  dừng: left=3, right=2; đáp án = 5.0


Quan sát: mỗi vòng `left` tăng và `right` giảm đúng `1`, nên sau `n/2` vòng chúng "vượt qua nhau" — vừa đủ dùng hết mọi phần tử, mỗi phần tử đúng một lần.

## 9. Code nộp LeetCode

Bản nộp dùng Cách 3 (sắp xếp rồi `min` các tổng đối xứng): ngắn, `O(n log n)`.

In [18]:
class Solution:
    def minimumAverage(self, nums: List[int]) -> float:
        nums.sort()                      # nhỏ nhất ở đầu, lớn nhất ở cuối
        n = len(nums)
        # bước k xoá nums[k] và nums[n - 1 - k]; tổng nhỏ nhất <=> trung bình nhỏ nhất
        return min(nums[i] + nums[n - 1 - i] for i in range(n // 2)) / 2


sol = Solution()
for nums, expected in examples:
    print(nums, "->", sol.minimumAverage(list(nums)))

[7, 8, 3, 4, 15, 13, 4, 1] -> 5.5
[1, 9, 8, 3, 10, 5] -> 5.5
[1, 2, 3, 7, 8, 9] -> 5.0


## 10. Vì sao thuật toán đúng

- **Min và max ở hai đầu mảng đã sắp xếp.** Ở vòng `1`, min là `nums[0]` và max là `nums[n-1]`.
- **Quy nạp:** sau `k` vòng ta đã xoá `k` phần tử nhỏ nhất `nums[0..k-1]` và `k` phần tử lớn nhất `nums[n-k..n-1]`. Phần còn lại là `nums[k..n-k-1]` (vẫn đã sắp xếp), nên vòng `k+1` lấy `nums[k]` và `nums[n-k-1]`. Vậy cặp ở vòng `i+1` là `(nums[i], nums[n-1-i])`.
- **Số vòng:** `n/2` vì mỗi vòng xoá hai phần tử và `n` chẵn; `i` chạy `0..n/2-1`, các cặp không chạm nhau.
- **Giá trị trùng nhau:** nếu có nhiều phần tử bằng nhau thì "xoá một số nhỏ nhất" hay "xoá số nhỏ nhất khác" đều cho cùng **giá trị**, nên kết quả không đổi (mảng đã sắp xếp xếp chúng cạnh nhau).
- **Chia `2` một lần:** `x → x / 2` tăng ngặt, nên `min` của trung bình = `min` của tổng rồi chia `2`.
- **Kiểu số:** `/` trong Python 3 luôn trả `float`; các giá trị `k / 2` biểu diễn chính xác nên không sai số.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(nums)` (`n <= 50`).

| Cách | Time | Space |
|---|---|---|
| 1. Mô phỏng `min`/`max`/`remove` | `O(n²)` | `O(n)` |
| 2. Sắp xếp + hai con trỏ + `averages` | `O(n log n)` | `O(n)` |
| 3. Sắp xếp + `min` tổng | `O(n log n)` | `O(1)` phụ (sort tại chỗ) |
| 4. `deque` sau sắp xếp | `O(n log n)` | `O(n)` |
| 5. Đếm theo giá trị | `O(n + V)`, `V = 51` | `O(n + V)` |

Với `n <= 50` mọi cách đều tức thì; cách 3 là gọn nhất.

## 12. Bộ test

Ta gom các cách vào `impls` và so với kỳ vọng.

In [19]:
sol = Solution()
impls = {
    "simulate": minimum_average_simulate,
    "two_pointers": minimum_average_two_pointers,
    "min_sum": minimum_average_min_sum,
    "deque": minimum_average_deque,
    "counting": minimum_average_counting,
    "leetcode": lambda a: sol.minimumAverage(list(a)),
}


def check_all(nums, expected, label):
    for key, fn in impls.items():
        actual = fn(list(nums))
        assert actual == expected, (label, key, nums, actual, expected)
        assert isinstance(actual, float), (label, key, type(actual))
    print("PASS", label)


for idx, (nums, expected) in enumerate(examples, 1):
    check_all(nums, expected, f"ví dụ {idx}")

PASS ví dụ 1
PASS ví dụ 2
PASS ví dụ 3


### 12.1. Các trường hợp biên

In [20]:
check_all([1, 1], 1.0, "hai phần tử bằng nhau")
check_all([1, 2], 1.5, "hai phần tử, trung bình .5")
check_all([50, 50], 50.0, "giá trị lớn nhất")
check_all([1, 50], 25.5, "nhỏ nhất và lớn nhất của miền")
check_all([5, 5, 5, 5], 5.0, "toàn bằng nhau")
check_all([1, 1, 50, 50], 25.5, "hai cặp giống nhau")
check_all([4, 3, 2, 1], 2.5, "đã sắp xếp giảm dần")
check_all([1, 2, 3, 4], 2.5, "đã sắp xếp tăng dần")
check_all([10, 1, 10, 1], 5.5, "chỉ hai giá trị")
check_all([1, 2, 49, 50], 25.5, "(1,50) và (2,49) cùng 25.5")
check_all([1, 3, 5, 7, 9, 11, 13, 15], 8.0, "dãy cách đều: mọi cặp cùng tổng 16")
check_all(list(range(1, 51)), 25.5, "1..50: mọi cặp có tổng 51")
check_all([1] * 25 + [50] * 25, 25.5, "25 số 1 và 25 số 50")

PASS hai phần tử bằng nhau
PASS hai phần tử, trung bình .5
PASS giá trị lớn nhất
PASS nhỏ nhất và lớn nhất của miền
PASS toàn bằng nhau
PASS hai cặp giống nhau
PASS đã sắp xếp giảm dần
PASS đã sắp xếp tăng dần
PASS chỉ hai giá trị
PASS (1,50) và (2,49) cùng 25.5
PASS dãy cách đều: mọi cặp cùng tổng 16
PASS 1..50: mọi cặp có tổng 51
PASS 25 số 1 và 25 số 50


### 12.2. Phản ví dụ cho các cách làm sai

Ba lỗi phổ biến: (a) ghép **sai cặp** `nums[i]` với `nums[i + n/2]` (nửa trái với nửa phải theo cùng thứ tự); (b) lấy **`max`** của trung bình thay vì `min`; (c) dùng **`//`** (mất `.5`).

In [21]:
def wrong_pair_halves(nums):
    nums = sorted(nums)
    n = len(nums)
    return min(nums[i] + nums[i + n // 2] for i in range(n // 2)) / 2     # SAI: ghép nhỏ i với trung vị i


def wrong_max(nums):
    nums = sorted(nums)
    n = len(nums)
    return max(nums[i] + nums[n - 1 - i] for i in range(n // 2)) / 2      # SAI: đề cần min


def wrong_floor_div(nums):
    nums = sorted(nums)
    n = len(nums)
    return min(nums[i] + nums[n - 1 - i] for i in range(n // 2)) // 2     # SAI: // cắt .5


tests = [[1, 2, 3, 7, 8, 9], [7, 8, 3, 4, 15, 13, 4, 1], [1, 2]]
for t in tests:
    print(t, "| đúng:", Solution().minimumAverage(list(t)), "| nửa-nửa:", wrong_pair_halves(t), "| max:", wrong_max(t), "| //:", wrong_floor_div(t))
assert wrong_pair_halves([1, 2, 3, 7, 8, 9]) != Solution().minimumAverage([1, 2, 3, 7, 8, 9])
assert wrong_max([7, 8, 3, 4, 15, 13, 4, 1]) != Solution().minimumAverage([7, 8, 3, 4, 15, 13, 4, 1])
assert wrong_floor_div([1, 2]) != Solution().minimumAverage([1, 2])

[1, 2, 3, 7, 8, 9] | đúng: 5.0 | nửa-nửa: 4.0 | max: 5.0 | //: 5
[7, 8, 3, 4, 15, 13, 4, 1] | đúng: 5.5 | nửa-nửa: 4.0 | max: 8.0 | //: 5
[1, 2] | đúng: 1.5 | nửa-nửa: 1.5 | max: 1.5 | //: 1


### 12.3. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

Sinh `nums` có độ dài **chẵn** trong `2..50`, giá trị `1..50`; trộn các kiểu: miền rộng, miền hẹp (nhiều trùng), chỉ vài giá trị. Mọi cách phải khớp `minimum_average_simulate` (làm đúng từng chữ đề). Thêm tính chất: đáp án nằm trong `[min, max]` của `nums` và là bội của `0.5`. Cuối cùng vét cạn **mọi** mảng độ dài `2, 4` với giá trị `1..8`.

In [22]:
import itertools

rng = random.Random(3194)
for trial in range(3000):
    n = rng.choice(range(2, 51, 2))
    mode = rng.choice(["wide", "narrow", "few"])
    if mode == "wide":
        nums = [rng.randint(1, 50) for _ in range(n)]
    elif mode == "narrow":
        nums = [rng.randint(20, 24) for _ in range(n)]
    else:
        vals = [rng.randint(1, 50) for _ in range(3)]
        nums = [rng.choice(vals) for _ in range(n)]
    assert 2 <= len(nums) <= 50 and len(nums) % 2 == 0 and all(1 <= x <= 50 for x in nums)
    ref = minimum_average_simulate(nums)
    for key, fn in impls.items():
        assert fn(list(nums)) == ref, (key, nums)
    assert min(nums) <= ref <= max(nums) and (ref * 2) == int(ref * 2)
print("PASS 3000 mảng ngẫu nhiên độ dài chẵn đúng ràng buộc")

total = 0
for n in (2, 4):
    for nums in itertools.product(range(1, 9), repeat=n):
        ref = minimum_average_simulate(list(nums))
        for key, fn in impls.items():
            assert fn(list(nums)) == ref, (key, nums)
        total += 1
print("PASS vét cạn", total, "mảng độ dài 2 và 4 với giá trị 1..8")

PASS 3000 mảng ngẫu nhiên độ dài chẵn đúng ràng buộc


PASS vét cạn 4160 mảng độ dài 2 và 4 với giá trị 1..8


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `//` (chia nguyên) thay vì `/`

In [23]:
def sai_chia_nguyen(nums):
    nums = sorted(nums)
    n = len(nums)
    best = min(nums[i] + nums[n - 1 - i] for i in range(n // 2))
    return best // 2                     # SAI: 11 // 2 = 5, mất .5


print("sai:", sai_chia_nguyen([1, 9, 8, 3, 10, 5]), "| đúng:", Solution().minimumAverage([1, 9, 8, 3, 10, 5]))

sai: 5 | đúng: 5.5


**Vì sao sai:** trung bình của hai số nguyên có thể là `x.5`. `//` cắt bỏ phần thập phân nên `11 // 2 = 5` thay vì `5.5`. **Cách sửa:** dùng `/` để có `float`.

### Lỗi 2: Gọi `sorted(nums)` nhưng không gán lại (hoặc viết `nums = nums.sort()`)

In [24]:
def sai_khong_gan_lai(nums):
    sorted(nums)                         # SAI: sorted trả list mới nhưng ta bỏ đi, nums vẫn chưa sắp xếp
    n = len(nums)
    return min(nums[i] + nums[n - 1 - i] for i in range(n // 2)) / 2


def sai_gan_none(nums):
    nums = nums.sort()                   # SAI: sort() trả None
    return nums


print("sai:", sai_khong_gan_lai([1, 9, 8, 3, 10, 5]), "| đúng:", Solution().minimumAverage([1, 9, 8, 3, 10, 5]))
print("sai:", sai_gan_none([3, 1, 2]), "(None!)")

sai: 3.0 | đúng: 5.5
sai: None (None!)


**Vì sao sai:** `sorted(x)` **không đổi** `x`; `x.sort()` đổi `x` nhưng **trả về `None`**. Hai thói quen nên nhớ: `nums = sorted(nums)` hoặc `nums.sort()` (không gán). **Cách sửa:** một trong hai cách đó.

### Lỗi 3: Ghép sai cặp (`nums[i]` với `nums[i + 1]` hoặc `nums[i + n/2]`)

In [25]:
def sai_ghep_ke(nums):
    nums = sorted(nums)
    return min(nums[i] + nums[i + 1] for i in range(0, len(nums), 2)) / 2     # SAI: ghép hai số liền kề


print("sai:", sai_ghep_ke([1, 2, 3, 7, 8, 9]), "| đúng:", Solution().minimumAverage([1, 2, 3, 7, 8, 9]))

sai: 1.5 | đúng: 5.0


**Vì sao sai:** đề xoá **nhỏ nhất cùng lớn nhất**, tức hai số ở **hai đầu** (đối xứng), không phải hai số liền kề. Với `[1,2,3,7,8,9]` cặp đúng là `(1,9), (2,8), (3,7)` (mỗi cặp trung bình `5`), còn ghép liền kề cho `(1,2) → 1.5`. **Cách sửa:** `nums[i]` với `nums[n - 1 - i]`.

### Lỗi 4: Lấy `max` của các trung bình (hoặc quên `min`)

In [26]:
def sai_lay_max(nums):
    nums = sorted(nums)
    n = len(nums)
    return max((nums[i] + nums[n - 1 - i]) / 2 for i in range(n // 2))       # SAI: đề cần nhỏ nhất


print("sai:", sai_lay_max([7, 8, 3, 4, 15, 13, 4, 1]), "| đúng:", Solution().minimumAverage([7, 8, 3, 4, 15, 13, 4, 1]))

sai: 8.0 | đúng: 5.5


**Vì sao sai:** đề hỏi **phần tử nhỏ nhất** của `averages`. Đọc kỹ từ khoá "minimum": ví dụ 1 có `averages = [8, 8, 6, 5.5]` và đáp án `5.5`, không phải `8`. **Cách sửa:** `min(...)`.

*(Ghi chú: duyệt `i` qua **toàn bộ** `0..n-1` thay vì chỉ `0..n/2-1` **không sai** kết quả vì các cặp đối xứng lặp lại, chỉ tốn gấp đôi công.)*

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lần lượt lấy nhỏ nhất và lớn nhất" ⇒ **sắp xếp** rồi **hai con trỏ** từ hai đầu (cùng họ: ghép cặp đối xứng).
- Không cần mô phỏng `remove`: sau khi sắp xếp, bước `k` luôn là `nums[k]` và `nums[n-1-k]`.
- Trung bình chỉ là tổng chia `2`; so sánh bằng tổng rồi chia **một lần** cuối.
- Nhớ khác biệt `sorted` (list mới) / `sort()` (tại chỗ, trả `None`); và `/` (float) / `//` (nguyên).
- Giá trị nhỏ (`<= 50`) gợi ý **counting sort**, nhưng với `n <= 50` thì `sort` là đủ.
- Bài họ hàng: **Array Partition (561)**, **Boats to Save People (881)**, **Minimize Maximum Pair Sum in Array (1877)**: cùng kỹ thuật ghép nhỏ nhất với lớn nhất.

## 15. Biến thể của bài

1. **Maximum Average:** trả về phần tử **lớn nhất** của `averages` (đổi `min` thành `max`).
2. **Minimize Maximum Pair Sum in Array (1877):** ghép các số thành cặp để **tổng lớn nhất của một cặp là nhỏ nhất** — cùng kỹ thuật ghép đối xứng.
3. **Trả về toàn bộ `averages`** (theo thứ tự các vòng) thay vì chỉ min.
4. **`n` lẻ:** số còn lại cuối cùng ở giữa; quy ước tự đặt (ví dụ trung bình của chính nó).
5. **Boats to Save People (881)**, **Two Sum II (167):** hai con trỏ trên mảng đã sắp xếp.

In [27]:
def maximum_average(nums):
    nums = sorted(nums)
    n = len(nums)
    return max(nums[i] + nums[n - 1 - i] for i in range(n // 2)) / 2


def all_averages(nums):
    nums = sorted(nums)
    n = len(nums)
    return [(nums[i] + nums[n - 1 - i]) / 2 for i in range(n // 2)]


def min_max_pair_sum(nums):
    # LeetCode 1877: tổng cặp lớn nhất nhỏ nhất
    nums = sorted(nums)
    n = len(nums)
    return max(nums[i] + nums[n - 1 - i] for i in range(n // 2))


print(maximum_average([7, 8, 3, 4, 15, 13, 4, 1]))            # 8.0
print(all_averages([7, 8, 3, 4, 15, 13, 4, 1]))               # [8.0, 8.0, 6.0, 5.5]
print(min_max_pair_sum([3, 5, 2, 3]))                         # 7  (cặp (2,5) và (3,3))
assert maximum_average([7, 8, 3, 4, 15, 13, 4, 1]) == 8.0
assert all_averages([7, 8, 3, 4, 15, 13, 4, 1]) == [8.0, 8.0, 6.0, 5.5]
assert all_averages([1, 9, 8, 3, 10, 5]) == [5.5, 6.0, 6.5]
assert min_max_pair_sum([3, 5, 2, 3]) == 7
assert min_max_pair_sum([3, 5, 4, 2, 4, 6]) == 8

8.0
[8.0, 8.0, 6.0, 5.5]
7


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mô phỏng | `min`, `max`, `remove` mỗi vòng | `O(n²)` | `O(n)` | hiểu đề, đối chiếu |
| 2. Sắp xếp + hai con trỏ | cặp `(left, right)`, lưu `averages` | `O(n log n)` | `O(n)` | trực quan, dễ trace |
| 3. Sắp xếp + `min` tổng | `min(nums[i] + nums[n-1-i]) / 2` | `O(n log n)` | `O(1)` phụ | **nên nộp** |
| 4. `deque` | `popleft` + `pop` sau sắp xếp | `O(n log n)` | `O(n)` | bám sát đề "xoá" |
| 5. Đếm theo giá trị | counting sort (`V = 51`) | `O(n + V)` | `O(n + V)` | giá trị bị chặn nhỏ |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sorted([3, 1, 2])`, `11 // 2`, `[10, 20, 30, 40][::-1]`, `Solution().minimumAverage([1, 3])`, `Solution().minimumAverage([2, 2, 4, 4])`.

**Bài 2 (tính tay).** Với `nums = [6, 1, 9, 4, 2, 8]`: sắp xếp, liệt kê các cặp đối xứng, trung bình từng cặp và đáp án.

**Bài 3 (điền TODO).** Hoàn thành `minimum_average_todo(nums)` theo Cách 2 (hai con trỏ).

In [28]:
def minimum_average_todo(nums):
    nums = sorted(nums)
    averages = []
    left, right = 0, len(nums) - 1
    # TODO 1: while left < right:
    # TODO 2:     thêm (nums[left] + nums[right]) / 2 vào averages
    # TODO 3:     left += 1, right -= 1
    # TODO 4: trả về min(averages)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `minimum_average_zip(nums)` bằng `zip(nums, reversed(nums))` và `itertools.islice` (hoặc cắt list) để lấy đúng `n // 2` cặp đầu, **không** dùng chỉ số `n - 1 - i`.

**Bài 5 (biến thể / mini challenge).** (a) Viết `minimum_average_odd(nums)` cho `n` **lẻ**: sau `(n - 1) / 2` vòng còn một phần tử `m`; quy ước thêm `m` (chính nó) vào `averages`; trả về `min`. (b) Viết `second_minimum_average(nums)` trả về **trung bình nhỏ thứ hai** (phần tử nhỏ thứ hai của `averages`, tính cả trùng; nếu chỉ có một trung bình thì trả `None`).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, 2, 3]`, `5`, `[40, 30, 20, 10]`, `2.0`, `3.0` (`[2,2,4,4]` → `(2+4)/2 = 3`). Bài 3: vòng `while`, cập nhật hai con trỏ ở cuối mỗi vòng.
- **Mức 2 (vừa):** Bài 2: sau sắp xếp `[1, 2, 4, 6, 8, 9]`; cặp `(1,9) → 5.0`, `(2,8) → 5.0`, `(4,6) → 5.0` ⇒ đáp án `5.0`. Bài 4: `list(zip(nums, nums[::-1]))[: n // 2]` (sau khi sắp xếp).
- **Mức 3 (sâu):** Bài 5(a): với `n` lẻ, phần tử giữa `nums[n // 2]` đứng một mình ⇒ thêm `nums[n // 2]` vào `averages`. Bài 5(b): sắp xếp `averages` rồi lấy phần tử `[1]` nếu độ dài `>= 2`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [29]:
# Bài 1
assert sorted([3, 1, 2]) == [1, 2, 3]
assert 11 // 2 == 5
assert [10, 20, 30, 40][::-1] == [40, 30, 20, 10]
assert Solution().minimumAverage([1, 3]) == 2.0
assert Solution().minimumAverage([2, 2, 4, 4]) == 3.0

# Bài 2
nums2 = sorted([6, 1, 9, 4, 2, 8])
assert nums2 == [1, 2, 4, 6, 8, 9]
assert [(nums2[i] + nums2[5 - i]) / 2 for i in range(3)] == [5.0, 5.0, 5.0]
assert Solution().minimumAverage([6, 1, 9, 4, 2, 8]) == 5.0


# Bài 3
def minimum_average_todo_ans(nums):
    nums = sorted(nums)
    averages = []
    left, right = 0, len(nums) - 1
    while left < right:
        averages.append((nums[left] + nums[right]) / 2)
        left += 1
        right -= 1
    return min(averages)


for nums, expected in examples:
    assert minimum_average_todo_ans(nums) == expected


# Bài 4
def minimum_average_zip(nums):
    nums = sorted(nums)
    pairs = list(zip(nums, reversed(nums)))[: len(nums) // 2]
    return min(a + b for a, b in pairs) / 2


for nums, expected in examples:
    assert minimum_average_zip(nums) == expected


# Bài 5(a)
def minimum_average_odd(nums):
    nums = sorted(nums)
    n = len(nums)
    averages = [(nums[i] + nums[n - 1 - i]) / 2 for i in range(n // 2)]
    if n % 2 == 1:
        averages.append(float(nums[n // 2]))         # phần tử giữa đứng một mình
    return min(averages)


assert minimum_average_odd([1, 2, 3, 7, 8, 9]) == 5.0       # n chẵn vẫn đúng như bài gốc
assert minimum_average_odd([5]) == 5.0
assert minimum_average_odd([1, 4, 10]) == 4.0               # (1+10)/2 = 5.5 và phần tử giữa 4 -> min = 4
assert minimum_average_odd([2, 3, 4, 5, 6]) == 4.0          # 4, 4 và phần tử giữa 4 -> 4.0


# Bài 5(b)
def second_minimum_average(nums):
    nums = sorted(nums)
    n = len(nums)
    averages = sorted((nums[i] + nums[n - 1 - i]) / 2 for i in range(n // 2))
    return averages[1] if len(averages) >= 2 else None


assert second_minimum_average([7, 8, 3, 4, 15, 13, 4, 1]) == 6.0     # averages sắp xếp [5.5, 6, 8, 8]
assert second_minimum_average([1, 2, 3, 7, 8, 9]) == 5.0
assert second_minimum_average([1, 2]) is None
print("PASS toàn bộ đáp án")

PASS toàn bộ đáp án


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (sắp xếp rồi ghép nhỏ nhất với lớn nhất)

```python
class Solution:
    def minimumAverage(self, nums: List[int]) -> float:
        nums.sort()
        n = len(nums)
        return min(nums[i] + nums[n - 1 - i] for i in range(n // 2)) / 2
```

**Checklist**

- [ ] Đã **sắp xếp** (và gán lại nếu dùng `sorted`; `nums.sort()` thì không gán)?
- [ ] Ghép **đối xứng** `nums[i]` với `nums[n - 1 - i]` (hai đầu), `i` chạy `0..n/2-1`?
- [ ] Lấy **`min`** (không phải `max`) của các trung bình?
- [ ] Dùng `/` (kết quả `float`, có `.5`), không phải `//`?
- [ ] Đã thử `[7,8,3,4,15,13,4,1]` (`5.5`), `[1,9,8,3,10,5]` (`5.5`), `[1,2,3,7,8,9]` (`5.0`) và `n = 2`?
- [ ] Skeleton đúng `class Solution` / `def minimumAverage(self, nums: List[int]) -> float`?